# Gain-prior + waveshaper — coloration by *composition*, not additive synthesis

**Google Colab**: Runtime → **GPU**. Open via *File → Open notebook → GitHub*
(`5aola/Virtual-Analogue-Compressor-Modelling`); cell 1 clones the repo for the
`06_output` modules and mounts Drive for the dataset. **Push local changes before running.**

## Idea — let harmonics come from a transfer curve, not a free waveform

The additive-color gain-prior run (`gain_prior_20260702_085618`) resolved the
*dynamics* side (test GR MAE 0.159 dB vs the SOTA's 0.314) but its waveform
metrics parked at the amplitude-match level (ESR ≈ 0.045 vs SOTA 0.0025): the
additive head `c = lin_c(h)` — a 33-param linear readout of the LSTM state —
must *synthesise* the ≈8 % RMS coloration residual as a free waveform, tracking
the signal's instantaneous phase at sample rate through 32 hidden units. It
learns ~nothing.

This notebook swaps the coloration mechanism to **composition**: the gained
signal is passed *through* a learnable memoryless waveshaper, so harmonics are
phase-locked and level-dependent by construction (a static I/O curve — what
VCA/output-stage saturation physically is). Dynamics stay in the prior + Δg +
LSTM; W only carries the curve (Wiener–Hammerstein-style factoring).

```
raw dry x ────────────────────────────────┐
x·g  (amplitude-matched, g = 10^(gr/20)) ─┤
gr   (reduction-positive, ~[0,1]) ────────┼─ cat → main LSTM(19→32)
tvcond cond_seq[16] (pool(|x|)⊕knobs) ────┘          │
                                   ┌─────────────────┴──────────────┐
                          Δg = 12·tanh(lin_g(h))          c = lin_c(h)  (optional)
                            (zero-init → 0 dB)          (zero-init → 0)
                                   │                           │
                     s = x · 10^((gr + Δg)/20)                 │
                     y = W(s) + c,   W(s) = s + r(s) − r(0)    │
                         r = tiny tanh MLP (zero-init output layer)
```

- **Identity at init is preserved**: W's output layer is zero-init, so at
  step 0 the output **is** the amplitude-matched signal — the same assert as
  the additive notebook passes unchanged.
- **W(0) = 0 by construction**: r(0) is subtracted through the same path, so
  the curve can never inject DC or signal into silence (the analogue
  counterpart: output coupling).
- **`WS_FILM=True`** (off by default): FiLM-modulate the curve's hidden layer
  from the LSTM state (zero-init → starts static) for programme-dependent
  bias-point drift. Run the static curve first.
- **8.4k params** (8.9k with FiLM) — still parameter-matched to the 8k SOTA
  LSTM32TVC (additive variant was 8.3k).

## Identical to the additive gain-prior notebook (ablation contract)
Dataset/split (seed 42), tvcond knob conditioning, 3 s crops, `batch_size=16`,
state reset per batch, TBPTT sub-steps of 4410, AdamW + cosine, fixed 100-epoch
budget, bf16 autocast, and the 4c loss (`0.5·L1 + 0.5·MR-STFT_ext +
0.2·envelope-dB + 0.1·pre-emphasis`). Reuses `system_gainprior.GainPriorSystem`
**unchanged** — the model's `return_parts` residual is (W(s)−s) + c, so the
logged `gain/*_color` reads "total non-gain output share" (cell 9 splits the
two). `model_gainprior.py` and `train_lstm_gain_prior.ipynb` are untouched and
keep working independently.

In [ ]:
# -- 0. Dependencies ---------------------------------------------------
# Uses nablafx (TVFiLMCond). Pin numpy first so lightning/nablafx installs
# can't downgrade Colab's numpy 2.x and break torch. Install lightning/nablafx
# --no-deps so they can't clobber Colab's CUDA torch. `rational` /
# `frechet_audio_distance` are nablafx import-chain deps we never use; stub
# both so `from nablafx...` doesn't drag in broken wheels.
!pip install -q "numpy>=2.0,<2.6"
!pip install -q torchmetrics soundfile auraloss einops lightning-utilities packaging
!pip install -q --no-deps lightning nablafx

import sys, types

rational = types.ModuleType("rational")
rational.torch = types.ModuleType("rational.torch")
rational.torch.Rational = type("Rational", (), {})
sys.modules["rational"], sys.modules["rational.torch"] = rational, rational.torch

fad = types.ModuleType("frechet_audio_distance")
fad.FrechetAudioDistance = type("FrechetAudioDistance", (), {})
sys.modules["frechet_audio_distance"] = fad

import numpy as np, torch
assert np.__version__.startswith("2."), f"numpy {np.__version__} - restart runtime, re-run cell 0"
print(f"numpy {np.__version__}, torch {torch.__version__}")

In [ ]:
# -- 1. Mount Drive (dataset) + clone repo from GitHub (code) ---------
# The repo is NOT synced to Drive (only data/ is). Code comes from GitHub -
# push local changes before (re)running this cell; re-running pulls updates.

import os
import sys
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive", force_remount=False)

DRIVE_DATA_ROOT = "/content/drive/Othercomputers/MacBook Air/data/Diff-SSL-G-Comp"
REPO_URL = "https://github.com/5aola/Virtual-Analogue-Compressor-Modelling.git"
REPO_ROOT = "/content/Virtual-Analogue-Compressor-Modelling"

if os.path.isdir(REPO_ROOT):
    !git -C "{REPO_ROOT}" fetch origin
    !git -C "{REPO_ROOT}" reset --hard origin/main
else:
    !git clone --depth 1 "{REPO_URL}" "{REPO_ROOT}"

DATA_ROOT = DRIVE_DATA_ROOT

# Module directory for this notebook (dataset_tfilm / model_gainprior_ws / ... live here).
COND_DIR = os.path.join(REPO_ROOT, "06_output")
assert os.path.isfile(os.path.join(COND_DIR, "model_gainprior_ws.py")), (
    f"Clone failed or stale: {COND_DIR}. Did you push local changes?"
)

OUTPUT_DIR = os.path.join(os.path.dirname(DATA_ROOT), "diffssl_gain_prior_runs")

assert os.path.isdir(os.path.join(DATA_ROOT, "gr_curves")), f"Bad DATA_ROOT: {DATA_ROOT}"
assert os.path.isdir(os.path.join(DATA_ROOT, "processed_ground_truth")), "Missing wet audio dir"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Drop cached local modules so a prior run cannot keep stale classes.
for _name in list(sys.modules):
    if _name in ("dataset_tfilm", "model_tfilm", "model_gainprior",
                 "model_gainprior_ws", "system_gainprior", "splits",
                 "amplitude_match"):
        del sys.modules[_name]

# repo root (for `src` + `nablafx`) + module dir
for p in (REPO_ROOT, os.path.join(REPO_ROOT, "nablafx"), COND_DIR):
    if p not in sys.path:
        sys.path.insert(0, p)

print(f"REPO_ROOT  : {REPO_ROOT}")
print(f"COND_DIR   : {COND_DIR}")
print(f"DATA_ROOT  : {DATA_ROOT}")
print(f"OUTPUT_DIR : {OUTPUT_DIR}")

In [ ]:
# -- 2. Cache dataset to Colab local SSD ------------------------------
# Same cache as the additive gain-prior notebook: dry WAV per song, plus
# gr_curve (.pt) + wet WAV per (song, setting) pair.

import shutil
from dataset_tfilm import discover_gr_pairs

LOCAL_DATA_ROOT = "/content/Diff-SSL-G-Comp"

pairs = discover_gr_pairs(DATA_ROOT)
settings = sorted({p["setting"] for p in pairs})
songs = sorted({p["song"] for p in pairs})
print(f"Caching {len(songs)} songs x {len(settings)} settings ({len(pairs)} pairs) -> {LOCAL_DATA_ROOT}")

def _mirror(src, dst):
    src, dst = Path(src), Path(dst)
    if not dst.exists() or dst.stat().st_size != src.stat().st_size:
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)

# dry WAVs (one per song, shared across settings)
for song in songs:
    fn = f"{song}_UnmasteredWAV.wav"
    _mirror(Path(DATA_ROOT) / "processed_normalized" / fn,
            Path(LOCAL_DATA_ROOT) / "processed_normalized" / fn)

# GR curves (.pt) + wet WAVs (-exported.wav), per (song, setting) pair
for p in pairs:
    _mirror(p["gr"], Path(LOCAL_DATA_ROOT) / "gr_curves" / p["setting"] / Path(p["gr"]).name)
    _mirror(p["wet"], Path(LOCAL_DATA_ROOT) / "processed_ground_truth" / p["setting"] / Path(p["wet"]).name)

DATA_ROOT = LOCAL_DATA_ROOT
print(f"Using local cache: {DATA_ROOT}")

In [ ]:
# -- 3. Imports & hyper-parameters (waveshaper gain-prior + 4c loss) --

import importlib
import json
from datetime import datetime

import torch
import lightning as pl
from lightning.pytorch.callbacks import (
    LearningRateMonitor, ModelCheckpoint, TQDMProgressBar,
)
from lightning.pytorch.loggers import CSVLogger, TensorBoardLogger

import dataset_tfilm as _dataset_tfilm
importlib.reload(_dataset_tfilm)
from dataset_tfilm import (
    BATCH_SIZE, SAMPLE_LENGTH, SAMPLE_RATE, GRCropDataModule, discover_gr_pairs,
)

import model_tfilm as _model_tfilm          # model_gainprior_ws imports from it
importlib.reload(_model_tfilm)
import model_gainprior_ws as _model_gainprior_ws
importlib.reload(_model_gainprior_ws)
from model_gainprior_ws import GainPriorWSDiffSSLLSTM

import system_gainprior as _system_gainprior   # reused UNCHANGED from the additive nb
importlib.reload(_system_gainprior)
from system_gainprior import GainPriorSystem

from splits import DIFFSSL_PARAM_RANGES, build_split_manifest
from src.dsp import PARAM_ORDER

print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "WARNING: CPU runtime")

# -- split (identical to 02b / the other 06_output notebooks) --
SPLIT_SEED   = 42
N_VAL_SONGS  = 1
N_TEST_SONGS = 2

# -- training (same diffssl TBPTT regime as the additive gain-prior nb) --
LR               = 1e-3
MAX_EPOCHS       = 100      # fixed budget == cosine T_max
STEP_NUM_SAMPLES = 4410     # diffssl TBPTT sub-step (0.1 s)
SCHEDULER        = "cosine"
ETA_MIN          = 1e-6
USE_AMP          = True
CHECK_VAL_EVERY_N_EPOCH = 1

# -- model core (identical to the additive gain-prior run) --
HIDDEN_SIZE     = 32
NUM_LAYERS      = 1
NUM_CONTROLS    = 4
TVCOND_DIM      = 16
COND_BLOCK_SIZE = 128
COND_NUM_LAYERS = 1
DELTA_MAX_DB    = 12.0   # bound on the learned gain correction (dB)
USE_COLOR       = True   # keep the additive head; the WS/color split in cell 9
                         # shows which mechanism actually carries the residual

# -- waveshaper (the one change vs the additive notebook) --
WS_HIDDEN = 8            # transfer-curve MLP width (1->8->8->1, ~96 params)
WS_FILM   = False        # True: FiLM the curve from the LSTM state (zero-init)

# -- loss (4c, unchanged). ENV_WEIGHT=0, PE_WEIGHT=0, MRSTFT_VARIANT="sota" == exact 02b --
TD_WEIGHT      = 0.5      # L1
FD_WEIGHT      = 0.5      # MR-STFT
ENV_WEIGHT     = 0.2      # RMS-envelope L1 in dB (differentiable GR MAE, window 1024)
PE_WEIGHT      = 0.1      # pre-emphasis L1 (transients)
MRSTFT_VARIANT = "extended"   # "extended" (adds 4096-FFT + lin-mag) | "sota"

RUN_TAG    = "diffssl_lstm32_gain_prior_ws"
RESUME_RUN = None

In [ ]:
# -- 4. Preview split (must match 02b / 05 / the other 06 notebooks) ---

preview = build_split_manifest(
    discover_gr_pairs(DATA_ROOT),
    seed=SPLIT_SEED, n_val_songs=N_VAL_SONGS, n_test_songs=N_TEST_SONGS,
)
print(f"Settings ({len(preview.all_settings)}): {preview.all_settings}")
print(f"Test settings (lowest T): {preview.test_settings}")
print(f"Train songs: {preview.train_songs}")
print(f"Val songs  : {preview.val_songs}")
print(f"Test songs : {preview.test_songs}")
print(f"Pairs - train={len(preview.train_pair_keys)} "
      f"val={len(preview.val_pair_keys)} test={len(preview.test_pair_keys)}")

In [ ]:
# -- 5. Model size ----------------------------------------------------

model = GainPriorWSDiffSSLLSTM(
    num_controls=NUM_CONTROLS, hidden_size=HIDDEN_SIZE, num_layers=NUM_LAYERS,
    tvcond_dim=TVCOND_DIM, cond_block_size=COND_BLOCK_SIZE, cond_num_layers=COND_NUM_LAYERS,
    delta_max_db=DELTA_MAX_DB, use_color=USE_COLOR,
    ws_hidden=WS_HIDDEN, ws_film=WS_FILM,
)
n_params = sum(p.numel() for p in model.parameters())
print(f"GainPriorWSDiffSSLLSTM: {n_params:,} params  "
      f"(hidden={HIDDEN_SIZE}, tvcond_dim={TVCOND_DIM}, controls={NUM_CONTROLS}, "
      f"delta_max={DELTA_MAX_DB} dB, color={USE_COLOR}, "
      f"ws_hidden={WS_HIDDEN}, ws_film={WS_FILM})")
for name, mod in model.named_children():
    print(f"  {name:10s} {sum(p.numel() for p in mod.parameters()):,}")
print(f"\nAdditive gain-prior was 8,322; SOTA LSTM32TVC is 8k - still parameter-matched.")
print(f"Crop {SAMPLE_LENGTH} ({SAMPLE_LENGTH/SAMPLE_RATE:.2f}s) | {SAMPLE_RATE} Hz | "
      f"TBPTT step {STEP_NUM_SAMPLES} | tvcond block {COND_BLOCK_SIZE}")

In [ ]:
# -- 6. DataModule + zero-init sanity check ---------------------------
# Zero-init linear heads + identity-init waveshaper make the untrained model
# IDENTICAL to the amplitude-matched baseline (dry x 10^(gr/20)). Verify on a
# real batch before training: the run must START from that baseline, not noise.

torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")

assert DATA_ROOT.startswith("/content/"), "Run the cache cell first (cell 2)."

NUM_WORKERS = min(8, os.cpu_count() or 2)
print(f"DataLoader num_workers: {NUM_WORKERS}")

if RESUME_RUN:
    RUN_NAME = RESUME_RUN
    RUN_DIR = os.path.join(OUTPUT_DIR, RUN_NAME)
    _resume_ckpt = os.path.join(RUN_DIR, "checkpoints", "last.ckpt")
    print(f"RESUMING: {RUN_NAME}")
else:
    RUN_NAME = f"gain_prior_ws_{datetime.now():%Y%m%d_%H%M%S}_{RUN_TAG}"
    RUN_DIR = os.path.join(OUTPUT_DIR, RUN_NAME)
    _resume_ckpt = None
    print(f"NEW run: {RUN_NAME}")

os.makedirs(RUN_DIR, exist_ok=True)
split_path = os.path.join(RUN_DIR, "split_manifest.json")

dm = GRCropDataModule(
    data_root=DATA_ROOT, sample_length=SAMPLE_LENGTH, sample_rate=SAMPLE_RATE,
    batch_size=BATCH_SIZE, split_seed=SPLIT_SEED,
    n_val_songs=N_VAL_SONGS, n_test_songs=N_TEST_SONGS,
    split_manifest_path=split_path, num_workers=NUM_WORKERS,
)
dm.setup()
print(f"Train/val/test crops: {len(dm.train_dataset)} / {len(dm.val_dataset)} / {len(dm.test_dataset)}")
print(f"Batches/epoch (train): {len(dm.train_dataloader())}  (batch_size={BATCH_SIZE})")

# -- zero-init sanity: untrained model == amplitude match --------------
from amplitude_match import amplitude_match

if not RESUME_RUN:
    _dry, _gr, _wet, _p = next(iter(dm.val_dataloader()))
    with torch.no_grad():
        model.reset_states()
        _y0 = model(_dry, _gr, _p)
    _diff = float((_y0 - amplitude_match(_dry, _gr)).abs().max())
    _l1 = float(torch.nn.functional.l1_loss(_y0, _wet))
    print(f"zero-init |model - amplitude_match| max = {_diff:.2e}")
    assert _diff < 1e-5, "gain-prior heads / waveshaper are not identity-initialised!"
    print(f"untrained (== amp-match) crop L1 vs wet: {_l1:.6f}  <- training starts here")
    model.reset_states()
    del _dry, _gr, _wet, _p, _y0

In [ ]:
# -- 7. Train ---------------------------------------------------------

with open(os.path.join(RUN_DIR, "hparams.json"), "w") as f:
    json.dump({
        "approach": "gain_prior_ws: y = W(x * 10^((gr + delta_g)/20)) + color, W identity-init",
        "model_type": "GainPriorWSDiffSSLLSTM",
        "model_ref": "additive gain-prior (gain_prior_20260702_085618) + waveshaper coloration stage",
        "dataset": "Diff-SSL-G-Comp", "setting": "multi (10 settings, tvcond on 4 knobs)",
        "conditioning": "knobs via tvcond (TVFiLMCond); GR as multiplicative prior input",
        "sample_rate": SAMPLE_RATE, "sample_length": SAMPLE_LENGTH, "batch_size": BATCH_SIZE,
        "step_num_samples": STEP_NUM_SAMPLES,
        "param_order": PARAM_ORDER, "param_ranges": DIFFSSL_PARAM_RANGES,
        "split_seed": SPLIT_SEED, "train_songs": dm.split.train_songs,
        "val_songs": dm.split.val_songs, "test_songs": dm.split.test_songs,
        "test_settings": dm.split.test_settings,
        "model": {"hidden_size": HIDDEN_SIZE, "num_layers": NUM_LAYERS,
                   "num_controls": NUM_CONTROLS, "tvcond_dim": TVCOND_DIM,
                   "cond_block_size": COND_BLOCK_SIZE, "cond_num_layers": COND_NUM_LAYERS,
                   "delta_max_db": DELTA_MAX_DB, "use_color": USE_COLOR,
                   "ws_hidden": WS_HIDDEN, "ws_film": WS_FILM,
                   "num_params": n_params},
        "loss": {"td_weight": TD_WEIGHT, "fd_weight": FD_WEIGHT,
                 "env_weight": ENV_WEIGHT, "pe_weight": PE_WEIGHT,
                 "mrstft_variant": MRSTFT_VARIANT,
                 "kind": "td*L1 + fd*MRSTFT + env*envdB_L1 + pe*preemph_L1"},
        "metrics": ["esr", "rmse", "mae", "mse"],
        "optimizer": f"adamw + {SCHEDULER}",
        "scheduler": SCHEDULER, "eta_min": ETA_MIN, "use_amp": USE_AMP,
        "check_val_every_n_epoch": CHECK_VAL_EVERY_N_EPOCH,
        "training": "diffssl_crop_batches + tbptt_substeps (reset each batch)",
        "lr": LR, "max_epochs": MAX_EPOCHS,
    }, f, indent=2)

system = GainPriorSystem(
    model=model, lr=LR, step_num_samples=STEP_NUM_SAMPLES,
    td_weight=TD_WEIGHT, fd_weight=FD_WEIGHT,
    env_weight=ENV_WEIGHT, pe_weight=PE_WEIGHT, mrstft_variant=MRSTFT_VARIANT,
    scheduler=SCHEDULER, max_epochs=MAX_EPOCHS, eta_min=ETA_MIN, use_amp=USE_AMP,
)

ckpt_dir = os.path.join(RUN_DIR, "checkpoints")
callbacks = [
    ModelCheckpoint(dirpath=ckpt_dir, monitor="loss/val", mode="min", save_top_k=3,
                    save_last=True, filename="best-{epoch:03d}-{step}",
                    auto_insert_metric_name=False),
    LearningRateMonitor(logging_interval="epoch"),
    TQDMProgressBar(refresh_rate=10),
]
loggers = [
    TensorBoardLogger(save_dir=RUN_DIR, name="tb", version=""),
    CSVLogger(save_dir=RUN_DIR, name="csv", version=""),
]

trainer = pl.Trainer(
    max_epochs=MAX_EPOCHS, accelerator="gpu", devices=1,
    callbacks=callbacks, logger=loggers, log_every_n_steps=10,
    check_val_every_n_epoch=CHECK_VAL_EVERY_N_EPOCH,
)
trainer.fit(system, dm, ckpt_path=_resume_ckpt)
print(f"Best val loss: {callbacks[0].best_model_score:.6f}")
print(f"Best ckpt    : {callbacks[0].best_model_path}")

In [ ]:
# -- 8. Test (held-out songs x lowest-threshold settings) -------------

best_ckpt = callbacks[0].best_model_path or os.path.join(ckpt_dir, "last.ckpt")
print(f"Testing with: {best_ckpt}")
trainer.test(system, datamodule=dm, ckpt_path=best_ckpt)

In [ ]:
# -- 9. Plot: prediction vs target + head shares ----------------------
# For each example: (top) waveform overlay, (bottom) GR input vs corrected
# gain gr+delta. Titles split the residual into the waveshaper share
# (mean|W(s)-s|, phase-locked harmonics) and the additive share (mean|color|,
# whatever cannot be a function of the instantaneous gained sample).

import matplotlib.pyplot as plt
import numpy as np
from system_gainprior import esr_metric

best = torch.load(callbacks[0].best_model_path, map_location="cuda", weights_only=False)
system.load_state_dict(best["state_dict"])
system.eval().cuda()
print(f"Loaded best checkpoint: {callbacks[0].best_model_path}")

val_batches = list(dm.val_dataloader())
dry, gr, wet, params = val_batches[len(val_batches) // 2]

with torch.no_grad():
    system.model.reset_states()
    pred, delta_db, color, ws_res = system.model(
        dry.cuda(), gr.cuda(), params.cuda(), return_parts_full=True)
    pred, delta_db = pred.cpu(), delta_db.cpu()
    color, ws_res = color.cpu(), ws_res.cpu()

dry_np, wet_np, pred_np = dry.numpy(), wet.numpy(), pred.numpy()
gr_np, delta_np = gr.numpy(), delta_db.numpy()
color_np, ws_np = color.numpy(), ws_res.numpy()
n_plots = min(3, dry_np.shape[0])
fig, axes = plt.subplots(2 * n_plots, 1, figsize=(14, 4.6 * n_plots), squeeze=False)
t = np.arange(wet_np.shape[-1]) / SAMPLE_RATE
for r in range(n_plots):
    ax = axes[2 * r, 0]
    ax.plot(t, dry_np[r, 0], label="Dry", alpha=0.35, lw=0.5, color="gray")
    ax.plot(t, wet_np[r, 0], label="Target (wet)", alpha=0.8, lw=0.5)
    ax.plot(t, pred_np[r, 0], label="Predicted", alpha=0.8, lw=0.5)
    pv = torch.from_numpy(pred_np[r]); tv = torch.from_numpy(wet_np[r])
    mae_r = float(np.mean(np.abs(pred_np[r, 0] - wet_np[r, 0])))
    ax.set_title(f"crop {r} - MAE {mae_r:.4f} | ESR {float(esr_metric(tv, pv)):.4f} | "
                 f"mean|dg| {np.abs(delta_np[r]).mean():.3f} dB | "
                 f"mean|ws| {np.abs(ws_np[r]).mean():.5f} | "
                 f"mean|color| {np.abs(color_np[r]).mean():.5f}")
    ax.set_ylabel("amp"); ax.legend(loc="lower right", fontsize=8); ax.set_ylim(-1.05, 1.05)

    ax = axes[2 * r + 1, 0]
    ax.plot(t, gr_np[r, 0], label="GR input (dB)", lw=0.7, color="#1f77b4")
    ax.plot(t, gr_np[r, 0] + delta_np[r, 0], label="GR + learned delta", lw=0.7,
            color="#d62728", alpha=0.8)
    ax.set_ylabel("gain (dB)"); ax.legend(loc="lower right", fontsize=8)
axes[-1, 0].set_xlabel("Time (s)")
fig.suptitle(f"Waveshaper gain-prior LSTM - best val loss {callbacks[0].best_model_score:.6f}", y=1.002)
fig.tight_layout()
plot_path = os.path.join(RUN_DIR, "eval_output_comparison.png")
fig.savefig(plot_path, dpi=150, bbox_inches="tight")
print(f"Saved plot -> {plot_path}")
plt.show()

In [ ]:
# -- 10. Learned transfer curve ---------------------------------------
# The waveshaper is where harmonics come from (composition): plot W(s)
# against identity plus the deviation. Should bow away from identity at high
# |s| (saturation); asymmetry = even harmonics. With WS_FILM=True this is the
# UNMODULATED static curve (h=None path).

s_sweep = torch.linspace(-1.0, 1.0, 1001).view(1, 1, -1).cuda()
with torch.no_grad():
    w_sweep = system.model.waveshaper(s_sweep).cpu().flatten().numpy()
s_np = s_sweep.cpu().flatten().numpy()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(s_np, s_np, "--", color="gray", lw=0.8, label="identity")
ax1.plot(s_np, w_sweep, color="#d62728", lw=1.2, label="W(s)")
ax1.set_xlabel("in"); ax1.set_ylabel("out"); ax1.set_title("Learned transfer curve")
ax1.legend(fontsize=8); ax1.grid(alpha=0.3)
ax2.plot(s_np, w_sweep - s_np, color="#d62728", lw=1.2)
ax2.set_xlabel("in"); ax2.set_ylabel("W(s) - s")
ax2.set_title(f"Deviation from identity (max {np.abs(w_sweep - s_np).max():.4f})")
ax2.grid(alpha=0.3)
fig.tight_layout()
curve_path = os.path.join(RUN_DIR, "eval_waveshaper_curve.png")
fig.savefig(curve_path, dpi=150, bbox_inches="tight")
print(f"Saved plot -> {curve_path}")
plt.show()

In [ ]:
%load_ext tensorboard
%tensorboard --logdir "{RUN_DIR}/tb"